In [1]:
import numpy as np
import tensorflow as tf

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.8
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from network import build_mlp_mixer_utkface_age
from train import WarmUpCosine, AdamW, LastNSaver, make_train_ds, make_test_ds, RSquared 

In [4]:
num_classes = 3
initial_lr = 5e-4
weight_decay = 1e-5
epochs = 200
warmup_epochs = 5
batch_size = 128
image_size = 32

In [ ]:
SAVE_PATH = "utkface_12k_split.npz" 
def load_dataset_if_exists(path=SAVE_PATH):
    """
    Load if file exists, otherwise return None.
    """
    if os.path.exists(path):
        data = np.load(path)
        print("✔ Cached data detected, loading directly")
        return (data["x_train"], data["y_train"],
                data["x_test"], data["y_test"])
    return None

In [6]:
x_train, y_train, x_test, y_test = load_dataset_if_exists()
y_train = tf.cast(y_train, dtype=tf.float32)
y_test = tf.cast(y_test, dtype=tf.float32)

✔ 检测到缓存数据，已直接加载


In [7]:
model = build_mlp_mixer_utkface_age(num_classes=1)

In [8]:
total_steps = epochs * (x_train.shape[0] // batch_size)
warmup_steps = warmup_epochs * (x_train.shape[0] // batch_size)
lr_schedule = WarmUpCosine(initial_lr, total_steps, warmup_steps)
optimizer = AdamW(
    learning_rate=lr_schedule,
    weight_decay=weight_decay
)
model.compile(
    optimizer=optimizer,
    loss='mse',
    metrics=[RSquared()]
)

In [9]:
saver = LastNSaver(n=20)

In [10]:
train_ds = make_train_ds(
    x_train, y_train,
    batch_size=batch_size,
    image_size=32, pad=4,
    mixup_alpha=0.05   # 0.2~0.4 common; increase if overfitting
)
test_ds = make_test_ds(x_test, y_test, batch_size=batch_size)

model.fit(
    train_ds,
    epochs=epochs,
    validation_data=test_ds,
    verbose=2,
    callbacks=[saver]
)

Epoch 1/200
156/156 - 17s - loss: 0.0322 - r_squared: -7.3337e-02 - val_loss: 0.0315 - val_r_squared: 0.0345 - 17s/epoch - 110ms/step
Epoch 2/200
156/156 - 8s - loss: 0.0244 - r_squared: 0.1895 - val_loss: 0.0251 - val_r_squared: 0.2311 - 8s/epoch - 54ms/step
Epoch 3/200
156/156 - 8s - loss: 0.0207 - r_squared: 0.3127 - val_loss: 0.0237 - val_r_squared: 0.2751 - 8s/epoch - 54ms/step
Epoch 4/200
156/156 - 8s - loss: 0.0182 - r_squared: 0.3954 - val_loss: 0.0179 - val_r_squared: 0.4525 - 8s/epoch - 54ms/step
Epoch 5/200
156/156 - 8s - loss: 0.0167 - r_squared: 0.4452 - val_loss: 0.0165 - val_r_squared: 0.4939 - 8s/epoch - 54ms/step
Epoch 6/200
156/156 - 8s - loss: 0.0159 - r_squared: 0.4752 - val_loss: 0.0142 - val_r_squared: 0.5652 - 8s/epoch - 54ms/step
Epoch 7/200
156/156 - 8s - loss: 0.0145 - r_squared: 0.5187 - val_loss: 0.0157 - val_r_squared: 0.5191 - 8s/epoch - 54ms/step
Epoch 8/200
156/156 - 8s - loss: 0.0139 - r_squared: 0.5350 - val_loss: 0.0132 - val_r_squared: 0.5947 - 8s/ep

In [11]:
model.save("Mix_8.h5")